## Metacritic

In [ ]:
# Cell 1: Import necessary libraries
import time
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
def get_available_platforms(game_base_url):
    """Finds all available platforms for a game."""
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=options
    )
    driver.get(game_base_url)

    # Get full page source
    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()

    platforms = []

    # Locate the dropdown menu containing platform options
    platform_select = soup.find("select", {"name": "Platforms"})
    if platform_select:
        for option in platform_select.find_all("option"):
            platform_slug = option.get(
                "value"
            ).strip()  # Get the URL-friendly platform name
            platform_name = option.text.strip()  # Get the readable platform name
            if platform_slug and platform_name:
                platforms.append((platform_name, platform_slug))

    return platforms

In [ ]:
# Cell 3: Define the function to scrape reviews
def get_all_metacritic_reviews(game_url, platform, review_type):
    """Scrapes all reviews for a given game, platform, and review type."""
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=options
    )
    driver.get(game_url)

    # Simulate scrolling to load all reviews
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.END)
        time.sleep(2)  # Allow time for loading

        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

    # Get full page source after scrolling
    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()

    # Extract review data
    reviews = []
    for review in soup.find_all("div", class_="c-siteReview"):
        score = (
            review.find("div", class_="c-siteReviewScore").find("span").text.strip()
            if review.find("div", class_="c-siteReviewScore")
            and review.find("div", class_="c-siteReviewScore").find("span")
            else "No Score"
        )
        date = (
            review.find("div", class_="c-siteReviewHeader_reviewDate").text.strip()
            if review.find("div", class_="c-siteReviewHeader_reviewDate")
            else "No Date"
        )
        text = (
            review.find("div", class_="c-siteReview_quote").find("span").text.strip()
            if review.find("div", class_="c-siteReview_quote")
            and review.find("div", class_="c-siteReview_quote").find("span")
            else "No Review Text"
        )
        if review_type == "user":
            username = (
                review.find("a", class_="c-siteReviewHeader_username").text.strip()
                if review.find("a", class_="c-siteReviewHeader_username")
                else "No Username"
            )
        else:
            username = (
                review.find(
                    "a", class_="c-siteReviewHeader_publicationName"
                ).text.strip()
                if review.find("a", class_="c-siteReviewHeader_publicationName")
                else "No Username"
            )
        reviews.append(
            {
                "platform": platform,
                "review_type": review_type,
                "username": username,
                "score": score,
                "date": date,
                "review": text,
            }
        )

    return reviews



In [ ]:
# Cell 4: Define the function to generate Metacritic URLs
def generate_metacritic_urls(game_names):
    """Generates Metacritic base URLs for each game."""
    base_url = "https://www.metacritic.com/game/"
    return {game: f"{base_url}{game}/" for game in game_names}


In [ ]:
# Cell 5: Define the main scraping function
def scrape_multiple_games(game_names, output_folder="metacritic_reviews"):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    game_urls = generate_metacritic_urls(game_names)

    for game, base_url in game_urls.items():
        print(f"Finding platforms for: {game}")
        platforms = get_available_platforms(f"{base_url}critic-reviews/")

        all_reviews = []
        for platform_name, platform_slug in platforms:
            critic_url = f"{base_url}critic-reviews/?platform={platform_slug}"
            user_url = f"{base_url}user-reviews/?platform={platform_slug}"

            print(f"Scraping {platform_name} - Critic Reviews for: {game}")
            try:
                reviews = get_all_metacritic_reviews(
                    critic_url, platform_name, "critic"
                )
                all_reviews.extend(reviews)
            except Exception as e:
                print(f"Error scraping {platform_name} critic reviews for {game}: {e}")

            print(f"Scraping {platform_name} - User Reviews for: {game}")
            try:
                reviews = get_all_metacritic_reviews(user_url, platform_name, "user")
                all_reviews.extend(reviews)
            except Exception as e:
                print(f"Error scraping {platform_name} user reviews for {game}: {e}")

        output_csv = os.path.join(output_folder, f"{game}.csv")

        df = pd.DataFrame(all_reviews)
        df.to_csv(output_csv, index=False, encoding="utf-8")

        if df.empty:
            print(
                f"No reviews found for {game}. CSV and XLSX created with only headers."
            )

        print(f"Saved {game} reviews to {output_csv}")



In [ ]:
# Cell 6: Define the list of games and run the scraper
mixed_game_names = [
    "cyberpunk_2077",
    "sea_of_thieves",
    "fallout_76",
    "total_war_rome_ii",
    "days_gone",
    "wildfrost",
    "final_fantasy_xiv_online",
    "warhammer_40000_darktide",
    "battlefield_2042",
    "wasteland_3",
]

scrape_multiple_games(mixed_game_names)

## Data Cleaning

In [ ]:
import glob
import pandas as pd
import os
import string
import emoji

In [ ]:
# CHANGE ACCORDINGLY: Define your mapping of CSV file names to game titles
CONSOLIDATED_BASE_DIR="/metacritic_reviews/CSV_files"
OUTPUT_DIR = "/content/drive/MyDrive/SC4021/Data/metacritic_reviews/Updated_CSV_files"

# CHANGE ACCORDINGLY: Define your mapping of CSV file names to game titles
# The key is the file name
game_titles = {
    'cyberpunk-2077': 'Cyberpunk 2077',
    'sea-of-thieves': 'Sea of Thieves',
    'Fallout-76': 'Fallout 76',
    'total-war-rome-ii': 'Total War: Rome 2',
    'battlefield-2': 'Battlefield 2',
    'days-gone': 'Days Gone',
    'wildfrost': 'Wildfrost',
    'final-fantasy-xiv-online': 'Final Fantasy 14',
    'warhammer-40000-darktide': 'Warhammer 40k: Darktide',
    'wasteland-3': 'Wasteland 3'
}

In [ ]:
def data_cleaning(base_dir, output_dir, game_titles, platform):
  # Use glob to get all CSV files in the directory
  csv_files = glob.glob(f"{base_dir}/*.csv")

  # Iterate over each CSV file
  for csv_file in csv_files:
      print(f"Processing {csv_file}")
      # to be used to add in column
      file_name = os.path.basename(csv_file).split('.')[0]

      # Load the CSV file into a pandas DataFrame
      df = pd.read_csv(csv_file)
      df = df.rename(columns={'platform': 'gaming_platform'})
      # CHANGE ACCORDINGLY: Add platform column
      df['platform'] = platform

      # Add game column
      # Check if the file name is in the game_titles dictionary
      if file_name in game_titles:
          # Add a new column 'Game' with the corresponding game title
          df['game'] = game_titles[file_name]
      else:
          print(f"Error: No game title found for {file_name}!")
      # Perform operations on the dataframe as needed
      # For example, printing the first few rows of the CSV
      print(df.head())
      # Save the updated dataframe to a new CSV file, just REWRITE
      output_file = os.path.join(output_dir, f"updated_{file_name}.csv")
      df.to_csv(output_file, index=False)
      print(f"Saved updated file: {output_file}")

  if platform != "Metacritic":
    csv_files = glob.glob(f"{output_dir}/*.csv")

    # Iterate over each CSV file
    for csv_file in csv_files:

        print(f"Processing {csv_file}")
        # to be used to add in column
        file_name = os.path.basename(csv_file).split('.')[0]

        # Load the CSV file into a pandas DataFrame
        df = pd.read_csv(csv_file)
        # Edit the timestamp of the row
        # Convert the 'timestamp' column to datetime format
        df['timestamp_updated'] = pd.to_datetime(df['timestamp_updated'])

        # Extract only the date part (removing the time) for the entire dataset
        df['timestamp_updated_date'] = df['timestamp_updated'].dt.date

        print(df.head())
        # Save the updated dataframe to a new CSV file, just REWRITE
        output_file = os.path.join(output_dir, f"{file_name}.csv")
        df.to_csv(output_file, index=False)
        print(f"Saved updated file: {output_file}")


In [ ]:
data_cleaning(CONSOLIDATED_BASE_DIR,OUTPUT_DIR,game_titles,"Metacritic")

In [ ]:
# Function to check the percentage of symbols in a string
def is_mostly_symbols(text, threshold=0.8):
    text = str(text)
    # Define what counts as a symbol (punctuation + other special characters)
    symbols = set(string.punctuation + '£$%^&*()_+=<>?{}[]|\\~`')

    # Count symbols and emojis
    symbol_count = sum(1 for char in text if char in symbols or emoji.is_emoji(char))

    # Calculate the percentage of symbols
    total_chars = len(text)
    if total_chars == 0:
        return False  # Handle empty strings

    symbol_ratio = symbol_count / total_chars

    # Return True if the symbol ratio exceeds the threshold
    return symbol_ratio > threshold


In [ ]:
# CHANGE ACCORDINGLY: Edit to your base directory, might need to create additional folder 'Updated_CSV_files'
UPDATED_BASE_DIR="/content/drive/MyDrive/SC4021/Data/metacritic_reviews/Updated_CSV_files"
csv_files = glob.glob(f"{UPDATED_BASE_DIR}/*.csv")

# Iterate over each CSV file
for csv_file in csv_files:

    print(f"Processing {csv_file}")
    # to be used to add in column
    file_name = os.path.basename(csv_file).split('.')[0]

    # Load the CSV file into a pandas DataFrame
    df = pd.read_csv(csv_file)
    # update df to remove those rows with a lot of symbols + empty
    updated_df = df[~df['review'].apply(is_mostly_symbols) & df['review'].str.strip().ne('').fillna(False)]
    print(f"--------Updated Length of {csv_file}: {len(updated_df)}--------")

    print(updated_df.head())
    # Save the updated dataframe to a new CSV file, just REWRITE
    # output_file = os.path.join(OUTPUT_DIR, f"RemovedSymbol_{file_name}.csv")
    output_file = os.path.join(OUTPUT_DIR, csv_file)
    updated_df.to_csv(output_file, index=False)
    print(f"Saved updated file: {output_file}")



In [ ]:
# Print updated length of datasets to compare

# Iterate over each CSV file
def sanitycheck(DIR):
    csv_files = glob.glob(f"{DIR}/*.csv")
    for csv_file in csv_files:

        # Load the CSV file into a pandas DataFrame
        df = pd.read_csv(csv_file)

        print(f"--------Updated Length of {csv_file}: {len(df)}--------")

In [ ]:
# AFTER DATA CLEANING
UPDATED_BASE_DIR="/content/drive/MyDrive/SC4021/Data/metacritic_reviews/Updated_CSV_files"
sanitycheck(UPDATED_BASE_DIR)

In [ ]:
# BEFORE DATA CLEANING
# Use glob to get all CSV files in the directory
sanitycheck(CONSOLIDATED_BASE_DIR)